# Check the trained model locally

Confirms that the model trained on Colab works on this machine, then tries it on groups of URLs
with `scripts/predict.py`. Run it with the project's `.venv` kernel, after downloading
`model/phishing_classifier.pth` and the `data/` splits from the Colab runtime.

The model only sees the hostname, so paths and query strings don't change the result.

In [ ]:
import json
import os
import shlex
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/predict.py").exists())
os.chdir(ROOT)
WEIGHTS = ROOT / "model/phishing_classifier.pth"

assert WEIGHTS.exists(), f"Missing {WEIGHTS}: download it from the Colab runtime into model/"
size_mb = WEIGHTS.stat().st_size / 1e6
assert size_mb > 400, f"{WEIGHTS.name} is only {size_mb:.0f} MB: the download looks incomplete"
print(f"Project: {ROOT}")
print(f"Python:  {sys.executable}")
print(f"Weights: {size_mb:.0f} MB")

PY = shlex.quote(sys.executable)


def as_args(urls):
    return " ".join(shlex.quote(u) for u in urls)

## Same results as on Colab?

Re-runs the real-world check locally and compares it with the metrics Colab saved. A match means
the weights downloaded intact and the local code matches what was trained.

In [ ]:
metrics_path = ROOT / "model/phishing_classifier.real_world_metrics.json"
colab_text = metrics_path.read_text()
colab = json.loads(colab_text)["metrics"]

!{PY} scripts/evaluate.py --split real_world

local = json.loads(metrics_path.read_text())["metrics"]
diffs = {k: (colab[k], local[k]) for k in colab if abs(colab[k] - local[k]) > 1e-3}
if diffs:
    metrics_path.write_text(colab_text)  # keep the Colab record; evaluate.py overwrote it
    print(f"Differs from the Colab run (colab, local): {diffs}")
else:
    print("Matches the Colab run")

## Everyday legitimate sites, including subdomains (the old model's weak spot)

In [ ]:
LEGIT = [
    "https://github.com/pytorch/pytorch/issues",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://mail.google.com/mail/u/0/#inbox",
    "https://accounts.google.com/signin",
    "https://docs.python.org/3/library/re.html",
    "https://login.microsoftonline.com/common/oauth2/authorize",
    "https://www.bbc.co.uk/news/technology",
    "https://cs.stanford.edu/people",
    "https://blog.cloudflare.com/",
    "https://www.google.com/search?q=weather",
    "https://zoom.us/j/1234567890",
]
!{PY} scripts/predict.py {as_args(LEGIT)}

## Made-up phishing-style hostnames: brand lookalikes, alarming words, free hosting

In [ ]:
SUSPICIOUS = [
    "https://paypal-account-verify-secure.com/signin",
    "https://amaz0n-billing-update.com/account",
    "https://microsoft-office365-verify.xyz/auth",
    "https://login-microsoftonline-auth.web.app/",
    "https://apple-icloud-locate-device.ga/",
    "http://paypal.com.secure-login.info/",
]
!{PY} scripts/predict.py {as_args(SUSPICIOUS)}

## Edge cases: one site spelled differently, IP addresses, input with no hostname

In [ ]:
EDGE = [
    "github.com",
    "HTTPS://WWW.GitHub.com/",
    "http://192.168.1.1/login",
    "https://paypal.com@evil-login.com/",
    "https://xn--pple-43d.com/",
    "https://www.",
]
!{PY} scripts/predict.py {as_args(EDGE)}

## Your own URLs

In [ ]:
MINE = [
    "https://example.com",
]
!{PY} scripts/predict.py {as_args(MINE)}